In [ ]:
import requests
import pandas as pd
from tqdm import trange
from tqdm import tqdm

In [ ]:
initiatives_url = "https://ec.europa.eu/info/law/better-regulation/brpapi/searchInitiatives?feedbackStatus=CLOSED&page={i}&size=100"
feedback_url = "https://ec.europa.eu/info/law/better-regulation/api/allFeedback?publicationId={id}&size=100&page={i}&sort=dateFeedback,DESC"
summary_url = "https://ec.europa.eu/info/law/better-regulation/brpapi/groupInitiatives/{id}"

In [ ]:
try:
    initiatives_df = pd.read_csv('initiatives_ids.csv')
except Exception:
    results = []
    for page_n in range(25):
        url=initiatives_url.format(i=page_n)
        r=requests.get(url)
        j = r.json()
        for init_n in range(len(j['_embedded']['initiativeResultDtoes'])):
            init_info=j['_embedded']['initiativeResultDtoes'][init_n]
            new_initiative={
                'id': init_info['id'],
                'reference': init_info['reference'],
                'shortTitle': init_info['shortTitle'],
                'foreseenActType': init_info.get('foreseenActType'),
                'topics': init_info.get('topics'),
                'feedbackStartDate': init_info['currentStatuses'][0].get('feedbackStartDate'),
                'feedbackEndDate': init_info['currentStatuses'][0].get('feedbackEndDate')
            }
            results.append(new_initiative)
    initiatives_df=pd.DataFrame(results)
    initiatives_df.to_csv('initiatives_ids.csv')

In [ ]:
initiatives_df

In [ ]:
# Fetching summaries
sum_results=[]
for init_n in trange(len(initiatives_df)):
    group_id = int(initiatives_df.iloc[init_n]['id'])
    r = requests.get(summary_url.format(id=group_id))
    j=r.json()
    for publ in j['publications']:
        initiative_info={
            'summary': j['dossierSummary'],
            'groupId': publ.get('groupId'),
            'id': publ.get('id'),
            'reference': publ.get('reference')
        }
        sum_results.append(initiative_info)

In [ ]:
INITIATIVES_PATH='initiatives_summary.csv'

try:
    summary_df=pd.read_csv(INITIATIVES_PATH)
except:
    summary_df=pd.DataFrame(sum_results)
    summary_df.to_csv(INITIATIVES_PATH)

In [ ]:
len(summary_df)

In [ ]:
import pickle

CACHE_PATH = 'feedbacks_cache.csv'
FAILED_PICKLE = 'failed_pages.pkl'
EMPTY_PICKLE = 'empty_pages.pkl'


try:
    failed_feedback_pages = pickle.load(open(FAILED_PICKLE, 'rb'))
    print(f"Loaded {len(failed_feedback_pages)} failed ids")
except:
    failed_feedback_pages=[]

try:
    empty_publications = pickle.load(open(EMPTY_PICKLE, 'rb'))
    print(f"Loaded {len(empty_publications)} empty ids")
except:
    empty_publications=[]

try:
    cache = pd.read_json(CACHE_PATH)
    cached_pub_ids = set(list(cache['pubId']))
    pub_ids_to_download = list(set(list(summary_df['id'])) - cached_pub_ids)
    print(f"Cached {len(cached_pub_ids)}, to download: {len(pub_ids_to_download)}")
except:
    print("No cache existing")
    cache=None
    pub_ids_to_download = list(summary_df['id'])

pub_ids_to_download = list(set(pub_ids_to_download)-set(failed_feedback_pages)-set(empty_publications))
print(f"Will download {len(pub_ids_to_download)}")

In [ ]:
import random
user_agents = [ 
	'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36', 
	'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36', 
	'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36', 
	'Mozilla/5.0 (iPhone; CPU iPhone OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148', 
	'Mozilla/5.0 (Linux; Android 11; SM-G960U) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.72 Mobile Safari/537.36' 
] 
def make_header():
    user_agent = random.choice(user_agents) 
    headers = {'User-Agent': user_agent} 
    return headers

In [ ]:
def save_to_cache(df):
    if len(df)!=0:
        df.index=df.id
        if cache is not None:
            df = pd.concat([cache, df])
            df.to_json(CACHE_PATH)
        else:
            df.to_json(CACHE_PATH)
        print(f"Saved to cache: {len(df)} rows")

In [ ]:
TIMEOUT = 5

feedbacks=[]
try:
    for i, pub_id in tqdm(enumerate(pub_ids_to_download), total=len(pub_ids_to_download)):
        header = make_header()
        r_page = requests.get(feedback_url.format(id=pub_id, i=0), headers=header, timeout=TIMEOUT)
        if r_page.status_code != 200:
            failed_feedback_pages.append(pub_id)
            continue
        pages_n = r_page.json()['page'].get('totalPages')
        if pages_n==0:
            empty_publications.append(pub_id)
        for page_no in range(pages_n):
            header = make_header()
            r = requests.get(feedback_url.format(id=pub_id, i=page_no), headers=header, timeout=TIMEOUT)
            publication_feedbacks=r.json()['_embedded']['feedback']
            for feedback in publication_feedbacks:
                feedback_info={
                    'pubId': pub_id,
                    'language': feedback.get('language'),
                    'id': feedback.get('id'),
                    'country': feedback.get('country'),
                    'organization': feedback.get('organization'),
                    'status': feedback.get('status'),
                    'surname': feedback.get('surname'),
                    'firstName': feedback.get('firstName'),
                    'feedback': feedback.get('feedback'),
                    'userType': feedback.get('userType'),
                    'companySize': feedback.get('companySize'),
                    'referenceInitiative': feedback.get('referenceInitiative'),
                    'publicationId': feedback.get('publicationId'),
                    'publicationStatus': feedback.get('publicationStatus'),
                    'dateFeedback': feedback.get('dateFeedback'),
                    'attachments': feedback.get('attachments')
                }
                feedbacks.append(feedback_info)
        if i%100==0:
            df = pd.DataFrame(feedbacks)
            save_to_cache(df)
except Exception as e:
    print(e)
    df = pd.DataFrame(feedbacks)
    if len(df)!=0:
        last_pub_id = df.iloc[-1]['pubId']
        df = df[df['pubId']!=last_pub_id]
    save_to_cache(df)
    pickle.dump(failed_feedback_pages, open(FAILED_PICKLE, 'wb'))
    pickle.dump(empty_publications, open(EMPTY_PICKLE, 'wb'))